In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

In [ ]:
df = pd.read_csv(
    'DataCoSupplyChainDataset.csv',
    encoding='latin1',
    low_memory=False
)

print("Dataset loaded successfully.")

In [ ]:
print(df.shape)
print(df.columns)

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("Dataset shape:", df.shape)


In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print("Column names:\n")

for i, column in enumerate(df.columns, start=1):
    print(i, column)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
print("Unique Order IDs:", df['Order Id'].nunique())
print("Total rows:", len(df))

In [ ]:
print(df['Late_delivery_risk'].value_counts())

In [ ]:
print(df['Late_delivery_risk'].value_counts(normalize=True) * 100)

In [ ]:
print("STEP 1 COMPLETE")
print("----------------------")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Unique Orders:", df['Order Id'].nunique())
print("Target:", 'Late_delivery_risk')

In [ ]:
# STEP 2A: Check missing values

missing_values = df.isnull().sum()

missing_percentage = (
    df.isnull().sum() / len(df) * 100
)

missing_table = pd.DataFrame({
    'Missing Values': missing_values,
    'Missing Percentage': missing_percentage
})

missing_table = missing_table[
    missing_table['Missing Values'] > 0
].sort_values(
    by='Missing Percentage',
    ascending=False
)

missing_table

In [ ]:
# STEP 2B: Check exact duplicate rows

duplicate_rows = df.duplicated().sum()

print("Number of exact duplicate rows:", duplicate_rows)

In [ ]:
# STEP 2C: Check repeated Order IDs

print("Total rows:", len(df))
print("Unique Order IDs:", df['Order Id'].nunique())

repeated_orders = df['Order Id'].duplicated().sum()

print("Repeated Order ID rows:", repeated_orders)

In [ ]:
order_item_counts = (
    df.groupby('Order Id')
      .size()
      .sort_values(ascending=False)
)

order_item_counts.head(10)

In [ ]:
order_item_counts.describe()

In [ ]:
# STEP 2D: Count unique values

unique_values = df.nunique().sort_values()

unique_values

In [ ]:
# STEP 2E: Find constant columns

constant_columns = [
    column for column in df.columns
    if df[column].nunique(dropna=False) <= 1
]

print("Constant columns:")
print(constant_columns)

In [ ]:
# STEP 2F: Check data types

data_types = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values
})

data_types

In [ ]:
# STEP 2G: Identify numerical and categorical columns

numerical_columns = df.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_columns = df.select_dtypes(
    include=['object']
).columns.tolist()

print("Number of numerical columns:", len(numerical_columns))
print("Number of categorical columns:", len(categorical_columns))

In [ ]:
print("\nNumerical Columns:")
for column in numerical_columns:
    print(column)
    print("\nCategorical Columns:")
for column in categorical_columns:
    print(column)

In [ ]:
# STEP 2H: Check target quality

print("Missing target values:")
print(df['Late_delivery_risk'].isnull().sum())

print("\nTarget unique values:")
print(df['Late_delivery_risk'].unique())

print("\nTarget counts:")
print(df['Late_delivery_risk'].value_counts())

print("\nTarget percentages:")
print(
    df['Late_delivery_risk']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# STEP 2I: Check known potentially duplicated columns

column_pairs = [
    ('Customer Id', 'Order Customer Id'),
    ('Product Card Id', 'Order Item Cardprod Id'),
    ('Category Id', 'Product Category Id'),
    ('Benefit per order', 'Order Profit Per Order'),
    ('Sales per customer', 'Order Item Total'),
    ('Product Price', 'Order Item Product Price')
]

for col1, col2 in column_pairs:
    same_percentage = (df[col1] == df[col2]).mean() * 100

    print(
        f"{col1} vs {col2}: "
        f"{same_percentage:.2f}% identical"
    )

In [ ]:
# STEP 2J: Check relationship between Delivery Status and target

pd.crosstab(
    df['Delivery Status'],
    df['Late_delivery_risk'],
    margins=True
)

In [ ]:
# STEP 2K: Check relationship between actual shipping time and target

pd.crosstab(
    df['Days for shipping (real)'],
    df['Late_delivery_risk']
)

In [ ]:
shipping_relationship = pd.crosstab(
    df['Days for shipping (real)'],
    df['Late_delivery_risk'],
    normalize='index'
) * 100

shipping_relationship.round(2)

In [ ]:
# STEP 2L: Final data-quality summary

print("DATA QUALITY AUDIT")
print("=" * 50)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nExact duplicate rows:")
print(df.duplicated().sum())

print("\nUnique orders:")
print(df['Order Id'].nunique())

print("\nMissing target values:")
print(df['Late_delivery_risk'].isnull().sum())

print("\nColumns containing missing values:")
print((df.isnull().sum() > 0).sum())

print("\nConstant columns:")
print(constant_columns)

print("\nStep 2 completed successfully.")

In [ ]:
# STEP 3A: Create a working copy

df_clean = df.copy()

print("Original dataset shape:", df.shape)
print("Working copy shape:", df_clean.shape)

In [ ]:
# STEP 3B: Remove irrelevant columns

columns_to_remove = [
    'Product Description',
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Product Image',
    'Order Zipcode'
]

df_clean = df_clean.drop(
    columns=columns_to_remove,
    errors='ignore'
)

print("Shape after removing irrelevant columns:", df_clean.shape)

In [ ]:
# STEP 3C: Remove redundant columns

redundant_columns = [
    'Order Customer Id',
    'Order Item Cardprod Id',
    'Product Category Id',
    'Benefit per order',
    'Sales per customer',
    'Order Item Product Price'
]

df_clean = df_clean.drop(
    columns=redundant_columns,
    errors='ignore'
)

print("Shape after removing redundant columns:", df_clean.shape)

In [ ]:
# STEP 3D: Convert date columns to datetime

df_clean['order date (DateOrders)'] = pd.to_datetime(
    df_clean['order date (DateOrders)'],
    errors='coerce'
)

df_clean['shipping date (DateOrders)'] = pd.to_datetime(
    df_clean['shipping date (DateOrders)'],
    errors='coerce'
)

print(df_clean[
    ['order date (DateOrders)', 'shipping date (DateOrders)']
].dtypes)

In [ ]:
print(
    "Missing order dates:",
    df_clean['order date (DateOrders)'].isnull().sum()
)

print(
    "Missing shipping dates:",
    df_clean['shipping date (DateOrders)'].isnull().sum()
)

In [ ]:
# STEP 3E: Re-check missing values

remaining_missing = df_clean.isnull().sum()

remaining_missing = remaining_missing[
    remaining_missing > 0
].sort_values(ascending=False)

remaining_missing

In [ ]:
if 'Customer Zipcode' in df_clean.columns:
    df_clean['Customer Zipcode'] = df_clean['Customer Zipcode'].fillna(
        df_clean['Customer Zipcode'].median()
    )

In [ ]:
# STEP 3F: Check important numerical ranges

check_columns = [
    'Sales',
    'Order Item Total',
    'Order Item Discount',
    'Order Item Discount Rate',
    'Order Item Quantity',
    'Product Price',
    'Order Profit Per Order',
    'Order Item Profit Ratio',
    'Days for shipping (real)',
    'Days for shipment (scheduled)'
]

df_clean[check_columns].describe().T

In [ ]:
# Check negative values in selected financial columns

financial_columns = [
    'Sales',
    'Order Item Total',
    'Order Item Discount',
    'Product Price',
    'Order Profit Per Order'
]

for col in financial_columns:
    negative_count = (df_clean[col] < 0).sum()
    print(col, "negative values:", negative_count)

In [ ]:
# STEP 3H: Check quantity values

print(
    df_clean['Order Item Quantity']
    .value_counts()
    .sort_index()
)

In [ ]:
invalid_quantity = (
    df_clean['Order Item Quantity'] <= 0
).sum()

print("Invalid zero/negative quantities:", invalid_quantity)

In [ ]:
# STEP 3I: Check discount-rate range

print(
    df_clean['Order Item Discount Rate']
    .describe()
)

print(
    "Minimum discount rate:",
    df_clean['Order Item Discount Rate'].min()
)

print(
    "Maximum discount rate:",
    df_clean['Order Item Discount Rate'].max()
)

In [ ]:
# STEP 3J: Strip spaces from text columns

text_columns = df_clean.select_dtypes(
    include='object'
).columns

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

print("Text columns standardized.")

In [ ]:
# STEP 3K: Check duplicates after cleaning

print(
    "Exact duplicate rows after cleaning:",
    df_clean.duplicated().sum()
)

In [ ]:
# STEP 3L: Validate target after cleaning

print(
    df_clean['Late_delivery_risk']
    .value_counts(dropna=False)
)

print(
    df_clean['Late_delivery_risk']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# Columns that must not be used as predictive inputs

leakage_columns = [
    'Delivery Status',
    'Days for shipping (real)',
    'shipping date (DateOrders)'
]

print("Leakage columns identified:")
for col in leakage_columns:
    print("-", col)

In [ ]:
# STEP 3N: Final cleaning summary

print("STEP 3 CLEANING SUMMARY")
print("=" * 50)

print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)

print("\nMissing values remaining:")
print(df_clean.isnull().sum().sum())

print("\nExact duplicate rows:")
print(df_clean.duplicated().sum())

print("\nUnique orders:")
print(df_clean['Order Id'].nunique())

print("\nTarget missing values:")
print(df_clean['Late_delivery_risk'].isnull().sum())

print("\nStep 3 completed successfully.")

In [ ]:
df_clean

In [ ]:
# STEP 4A: Target distribution

target_counts = df_clean['Late_delivery_risk'].value_counts()

print(target_counts)

target_counts.plot(kind='bar')
plt.title('Late Delivery Risk Distribution')
plt.xlabel('Late Delivery Risk')
plt.ylabel('Number of Records')
plt.show()

In [ ]:
shipping_late = pd.crosstab(
    df_clean['Shipping Mode'],
    df_clean['Late_delivery_risk'],
    normalize='index'
) * 100

shipping_late

In [ ]:
shipping_late.plot(kind='bar')

plt.title('Late Delivery Risk by Shipping Mode')
plt.xlabel('Shipping Mode')
plt.ylabel('Percentage')
plt.xticks(rotation=45)
plt.show()

In [ ]:
market_late = pd.crosstab(
    df_clean['Market'],
    df_clean['Late_delivery_risk'],
    normalize='index'
) * 100

market_late

In [ ]:
market_late.plot(kind='bar')

plt.title('Late Delivery Risk by Market')
plt.xlabel('Market')
plt.ylabel('Percentage')
plt.xticks(rotation=45)
plt.show()

In [ ]:
region_late = (
    df_clean.groupby('Order Region')['Late_delivery_risk']
    .mean()
    .sort_values(ascending=False)
    * 100
)

region_late

In [ ]:
region_late.plot(kind='bar', figsize=(12,6))

plt.title('Late Delivery Percentage by Region')
plt.xlabel('Order Region')
plt.ylabel('Late Delivery %')
plt.xticks(rotation=90)
plt.show()

In [ ]:
region_late = (
    df_clean.groupby('Order Region')['Late_delivery_risk']
    .mean()
    .sort_values(ascending=False)
    * 100
)

region_late

In [ ]:
region_late.plot(kind='bar', figsize=(12,6))

plt.title('Late Delivery Percentage by Region')
plt.xlabel('Order Region')
plt.ylabel('Late Delivery %')
plt.xticks(rotation=90)
plt.show()

In [ ]:
segment_late = pd.crosstab(
    df_clean['Customer Segment'],
    df_clean['Late_delivery_risk'],
    normalize='index'
) * 100

segment_late

In [ ]:
sales_market = (
    df_clean.groupby('Market')['Sales']
    .sum()
    .sort_values(ascending=False)
)

sales_market

In [ ]:
sales_market.plot(kind='bar')

plt.title('Total Sales by Market')
plt.xlabel('Market')
plt.ylabel('Total Sales')
plt.xticks(rotation=45)
plt.show()

In [ ]:
sales_category = (
    df_clean.groupby('Category Name')['Sales']
    .sum()
    .sort_values(ascending=False)
)

sales_category.head(15)

In [ ]:
sales_category.head(15).plot(kind='bar', figsize=(12,6))

plt.title('Top 15 Categories by Sales')
plt.xlabel('Category')
plt.ylabel('Total Sales')
plt.xticks(rotation=90)
plt.show()

In [ ]:
profit_category = (
    df_clean.groupby('Category Name')['Order Profit Per Order']
    .sum()
    .sort_values(ascending=False)
)

profit_category.head(15)

In [ ]:
profit_category.head(15).plot(kind='bar', figsize=(12,6))

plt.title('Top Categories by Profit')
plt.xlabel('Category')
plt.ylabel('Total Profit')
plt.xticks(rotation=90)
plt.show()

In [ ]:
plt.scatter(
    df_clean['Order Item Discount'],
    df_clean['Order Profit Per Order'],
    alpha=0.2
)

plt.title('Discount vs Profit')
plt.xlabel('Discount')
plt.ylabel('Profit')
plt.show()

In [ ]:
df_clean['order date (DateOrders)'] = pd.to_datetime(
    df_clean['order date (DateOrders)'],
    errors='coerce'
)

In [ ]:
# STEP 6: Date features

df_clean['Order_Year'] = df_clean[
    'order date (DateOrders)'
].dt.year

df_clean['Order_Month'] = df_clean[
    'order date (DateOrders)'
].dt.month

df_clean['Order_DayOfWeek'] = df_clean[
    'order date (DateOrders)'
].dt.dayofweek

df_clean['Order_Hour'] = df_clean[
    'order date (DateOrders)'
].dt.hour

df_clean['Order_Weekend'] = (
    df_clean['Order_DayOfWeek'] >= 5
).astype(int)

print(
    df_clean[
        [
            'Order_Year',
            'Order_Month',
            'Order_DayOfWeek',
            'Order_Hour',
            'Order_Weekend'
        ]
    ].head()
)

In [ ]:
# STEP 7: Create order-level modelling dataset

order_level = df_clean.groupby('Order Id').agg({

    'Late_delivery_risk': 'first',

    'Type': 'first',
    'Customer Segment': 'first',
    'Customer Country': 'first',
    'Customer State': 'first',

    'Market': 'first',
    'Order Country': 'first',
    'Order Region': 'first',

    'Shipping Mode': 'first',

    'Order Item Quantity': 'sum',
    'Sales': 'sum',
    'Order Item Discount': 'sum',
    'Order Item Discount Rate': 'mean',

    'Order Profit Per Order': 'sum',
    'Order Item Profit Ratio': 'mean',

    'Order_Year': 'first',
    'Order_Month': 'first',
    'Order_DayOfWeek': 'first',
    'Order_Hour': 'first',
    'Order_Weekend': 'first'

}).reset_index()

In [ ]:
item_counts = (
    df_clean.groupby('Order Id')
    .size()
    .reset_index(name='Number_of_Items')
)

order_level = order_level.merge(
    item_counts,
    on='Order Id',
    how='left'
)

In [ ]:
product_counts = (
    df_clean.groupby('Order Id')['Product Card Id']
    .nunique()
    .reset_index(name='Number_of_Products')
)

order_level = order_level.merge(
    product_counts,
    on='Order Id',
    how='left'
)

In [ ]:
category_counts = (
    df_clean.groupby('Order Id')['Category Id']
    .nunique()
    .reset_index(name='Number_of_Categories')
)

order_level = order_level.merge(
    category_counts,
    on='Order Id',
    how='left'
)

In [ ]:
print("Order-level shape:", order_level.shape)

order_level.head()

In [ ]:
print(
    order_level['Late_delivery_risk']
    .value_counts()
)

print(
    order_level['Late_delivery_risk']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
feature_columns = [

    'Type',
    'Customer Segment',
    'Customer Country',
    'Customer State',

    'Market',
    'Order Country',
    'Order Region',

    'Shipping Mode',

    'Order Item Quantity',
    'Sales',
    'Order Item Discount',
    'Order Item Discount Rate',
    'Order Profit Per Order',
    'Order Item Profit Ratio',

    'Number_of_Items',
    'Number_of_Products',
    'Number_of_Categories',

    'Order_Month',
    'Order_DayOfWeek',
    'Order_Hour',
    'Order_Weekend'
]

In [ ]:
X = order_level[feature_columns]

y = order_level['Late_delivery_risk']

In [ ]:
train_mask = order_level['Order_Year'] < 2018
test_mask = order_level['Order_Year'] == 2018

In [ ]:
X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

In [ ]:
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
categorical_features = [
    'Type',
    'Customer Segment',
    'Customer Country',
    'Customer State',
    'Market',
    'Order Country',
    'Order Region',
    'Shipping Mode'
]

In [ ]:
numerical_features = [
    'Order Item Quantity',
    'Sales',
    'Order Item Discount',
    'Order Item Discount Rate',
    'Order Profit Per Order',
    'Order Item Profit Ratio',

    'Number_of_Items',
    'Number_of_Products',
    'Number_of_Categories',

    'Order_Month',
    'Order_DayOfWeek',
    'Order_Hour',
    'Order_Weekend'
]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.impute import SimpleImputer

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

In [ ]:
categorical_transformer = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(strategy='most_frequent')
        ),

        (
            'encoder',
            OneHotEncoder(
                handle_unknown='ignore'
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            numeric_transformer,
            numerical_features
        ),

        (
            'cat',
            categorical_transformer,
            categorical_features
        )
    ]
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

In [ ]:
from sklearn.linear_model import LogisticRegression
logistic_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),

        (
            'classifier',
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

In [ ]:
logistic_model.fit(
    X_train,
    y_train
)

logistic_pred = logistic_model.predict(X_test)

logistic_prob = logistic_model.predict_proba(
    X_test
)[:, 1]

print(
    classification_report(
        y_test,
        logistic_pred
    )
)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
decision_tree_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),

        (
            'classifier',
            DecisionTreeClassifier(
                max_depth=8,
                min_samples_leaf=20,
                random_state=42
            )
        )
    ]
)

decision_tree_model.fit(
    X_train,
    y_train
)

tree_pred = decision_tree_model.predict(
    X_test
)

tree_prob = decision_tree_model.predict_proba(
    X_test
)[:, 1]

In [ ]:
from sklearn.ensemble import RandomForestClassifier
random_forest_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),

        (
            'classifier',
            RandomForestClassifier(
                n_estimators=200,
                max_depth=12,
                min_samples_leaf=10,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

random_forest_model.fit(
    X_train,
    y_train
)

rf_pred = random_forest_model.predict(
    X_test
)

rf_prob = random_forest_model.predict_proba(
    X_test
)[:, 1]

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gradient_model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),

        (
            'classifier',
            GradientBoostingClassifier(
                random_state=42
            )
        )
    ]
)

gradient_model.fit(
    X_train,
    y_train
)

gradient_pred = gradient_model.predict(
    X_test
)

gradient_prob = gradient_model.predict_proba(
    X_test
)[:, 1]

In [ ]:
def evaluate_model(
    name,
    y_true,
    y_pred,
    y_prob
):

    return {
        'Model': name,

        'Accuracy':
            accuracy_score(
                y_true,
                y_pred
            ),

        'Precision':
            precision_score(
                y_true,
                y_pred
            ),

        'Recall':
            recall_score(
                y_true,
                y_pred
            ),

        'F1 Score':
            f1_score(
                y_true,
                y_pred
            ),

        'ROC AUC':
            roc_auc_score(
                y_true,
                y_prob
            )
    }

In [ ]:
results = []

results.append(
    evaluate_model(
        'Logistic Regression',
        y_test,
        logistic_pred,
        logistic_prob
    )
)

results.append(
    evaluate_model(
        'Decision Tree',
        y_test,
        tree_pred,
        tree_prob
    )
)

results.append(
    evaluate_model(
        'Random Forest',
        y_test,
        rf_pred,
        rf_prob
    )
)

results.append(
    evaluate_model(
        'Gradient Boosting',
        y_test,
        gradient_pred,
        gradient_prob
    )
)

In [ ]:
results_df = pd.DataFrame(results)

results_df
results_df.sort_values(
    by='ROC AUC',
    ascending=False
)

In [ ]:
print(
    confusion_matrix(
        y_test,
        logistic_pred
    )
)

print(
    confusion_matrix(
        y_test,
        tree_pred
    )
)

print(
    confusion_matrix(
        y_test,
        rf_pred
    )
)

In [ ]:
print("LOGISTIC REGRESSION")

print(
    classification_report(
        y_test,
        logistic_pred
    )
)

print("DECISION TREE")

print(
    classification_report(
        y_test,
        tree_pred
    )
)

print("RANDOM FOREST")

print(
    classification_report(
        y_test,
        rf_pred
    )
)

In [ ]:
fpr_log, tpr_log, _ = roc_curve(
    y_test,
    logistic_prob
)

fpr_tree, tpr_tree, _ = roc_curve(
    y_test,
    tree_prob
)

fpr_rf, tpr_rf, _ = roc_curve(
    y_test,
    rf_prob
)

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(
    fpr_log,
    tpr_log,
    label='Logistic Regression'
)

plt.plot(
    fpr_tree,
    tpr_tree,
    label='Decision Tree'
)

plt.plot(
    fpr_rf,
    tpr_rf,
    label='Random Forest'
)

plt.plot(
    [0,1],
    [0,1],
    linestyle='--'
)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.title('ROC Curve Comparison')

plt.legend()

plt.show()

In [ ]:
trained_preprocessor = (
    random_forest_model.named_steps[
        'preprocessor'
    ]
)

encoded_names = (
    trained_preprocessor
    .named_transformers_['cat']
    .named_steps['encoder']
    .get_feature_names_out(
        categorical_features
    )
)

all_feature_names = (
    numerical_features
    + list(encoded_names)
)

rf_classifier = (
    random_forest_model
    .named_steps['classifier']
)

importance = (
    rf_classifier.feature_importances_
)

feature_importance = pd.DataFrame({

    'Feature':
        all_feature_names,

    'Importance':
        importance
})

feature_importance = (
    feature_importance
    .sort_values(
        by='Importance',
        ascending=False
    )
)

feature_importance.head(20)

feature_importance.head(20).plot(
    x='Feature',
    y='Importance',
    kind='bar',
    figsize=(12,6)
)

plt.title(
    'Top 20 Predictors of Late Delivery'
)

plt.ylabel('Importance')

plt.xticks(rotation=90)

plt.show()

In [ ]:
prediction_results = order_level.loc[
    test_mask,
    [
        'Order Id',
        'Market',
        'Order Region',
        'Shipping Mode',
        'Late_delivery_risk'
    ]
].copy()

prediction_results[
    'Predicted_Late_Risk'
] = rf_pred

prediction_results[
    'Late_Risk_Probability'
] = rf_prob
prediction_results.head()

In [ ]:
prediction_results['Risk_Level'] = pd.cut(

    prediction_results[
        'Late_Risk_Probability'
    ],

    bins=[
        0,
        0.40,
        0.70,
        1.00
    ],

    labels=[
        'Low Risk',
        'Medium Risk',
        'High Risk'
    ],

    include_lowest=True
)
prediction_results[
    'Risk_Level'
].value_counts()

In [ ]:
prediction_results.to_csv(
    'DataCo_Late_Delivery_Predictions.csv',
    index=False
)
from google.colab import files

files.download(
    'DataCo_Late_Delivery_Predictions.csv'
)



In [ ]:
results_df.to_csv(
    'DataCo_Model_Comparison.csv',
    index=False
)
files.download(
    'DataCo_Model_Comparison.csv'
)


In [ ]:
 results_df.to_csv(
    'DataCo_Model_Comparison.csv',
    index=False
)

 files.download(
    'DataCo_Model_Comparison.csv'
)

In [ ]:
feature_importance.to_csv(
    'DataCo_Feature_Importance.csv',
    index=False
)
files.download(
    'DataCo_Feature_Importance.csv'
)

In [ ]:
import joblib
joblib.dump(
    random_forest_model,
    'DataCo_Late_Delivery_Model.pkl'
)
files.download(
    'DataCo_Late_Delivery_Model.pkl'
)

In [ ]:
df
df_clean
order_level
X
y
X_train
X_test
y_train
y_test
preprocessor
logistic_model
decision_tree_model
random_forest_model

In [ ]:
# STEP 32A: Add exact order date to order_level if it is not already present

order_dates = (
    df_clean.groupby('Order Id')['order date (DateOrders)']
    .first()
    .reset_index(name='Order_Date')
)

if 'Order_Date' not in order_level.columns:
    order_level = order_level.merge(
        order_dates,
        on='Order Id',
        how='left'
    )

print(order_level[['Order Id', 'Order_Date']].head())

 # STEP 32B: Sort orders chronologically

order_level = order_level.sort_values(
    'Order_Date'
).reset_index(drop=True)

print("Earliest order:", order_level['Order_Date'].min())
print("Latest order:", order_level['Order_Date'].max())

In [ ]:
order_level['Average_Sales_Per_Item'] = (
    order_level['Sales'] /
    order_level['Number_of_Items'].replace(0, np.nan)
)

In [ ]:
order_level['Average_Quantity_Per_Item'] = (
    order_level['Order Item Quantity'] /
    order_level['Number_of_Items'].replace(0, np.nan)
)

In [ ]:
order_level['Discount_to_Sales_Ratio'] = (
    order_level['Order Item Discount'] /
    order_level['Sales'].replace(0, np.nan)
)

In [ ]:
order_level['Profit_Margin'] = (
    order_level['Order Profit Per Order'] /
    order_level['Sales'].replace(0, np.nan)
)

In [ ]:
order_level['Order_Complexity'] = (
    order_level['Number_of_Products'] +
    order_level['Number_of_Categories']
)

In [ ]:
order_level.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

In [ ]:
advanced_features = [
    'Average_Sales_Per_Item',
    'Average_Quantity_Per_Item',
    'Discount_to_Sales_Ratio',
    'Profit_Margin',
    'Order_Complexity'
]

order_level[advanced_features].describe().T

In [ ]:
categorical_features = [
    'Type',
    'Customer Segment',
    'Customer Country',
    'Customer State',
    'Market',
    'Order Country',
    'Order Region',
    'Shipping Mode'
]

numerical_features = [
    'Order Item Quantity',
    'Sales',
    'Order Item Discount',
    'Order Item Discount Rate',
    'Order Profit Per Order',
    'Order Item Profit Ratio',

    'Number_of_Items',
    'Number_of_Products',
    'Number_of_Categories',

    'Order_Month',
    'Order_DayOfWeek',
    'Order_Hour',
    'Order_Weekend',

    'Average_Sales_Per_Item',
    'Average_Quantity_Per_Item',
    'Discount_to_Sales_Ratio',
    'Profit_Margin',
    'Order_Complexity'
]

feature_columns = (
    categorical_features +
    numerical_features
)

X = order_level[feature_columns]

y = order_level['Late_delivery_risk']

In [ ]:
train_mask = (
    order_level['Order_Date'].dt.year < 2018
)

test_mask = (
    order_level['Order_Date'].dt.year == 2018
)

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True))

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(strategy='median')
        ),
        (
            'scaler',
            StandardScaler()
        )
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            'imputer',
            SimpleImputer(strategy='most_frequent')
        ),
        (
            'encoder',
            OneHotEncoder(
                handle_unknown='ignore'
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            numeric_transformer,
            numerical_features
        ),
        (
            'cat',
            categorical_transformer,
            categorical_features
        )
    ]
)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

time_cv = TimeSeriesSplit(
    n_splits=5
)

print(time_cv)



In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    brier_score_loss
)

def advanced_evaluate_model(
    model_name,
    y_true,
    y_pred,
    y_prob
):
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(
            y_true,
            y_pred
        ),
        'Precision': precision_score(
            y_true,
            y_pred
        ),
        'Recall': recall_score(
            y_true,
            y_pred
        ),
        'F1': f1_score(
            y_true,
            y_pred
        ),
        'ROC_AUC': roc_auc_score(
            y_true,
            y_prob
        ),
        'PR_AUC': average_precision_score(
            y_true,
            y_prob
        ),
        'Brier_Score': brier_score_loss(
            y_true,
            y_prob
        )
    }



In [ ]:
!pip install -q xgboost

from xgboost import XGBClassifier

In [ ]:
xgb_model = Pipeline(
    steps=[
        (
            'preprocessor',
            preprocessor
        ),
        (
            'classifier',
            XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                eval_metric='logloss',
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
xgb_model.fit(
    X_train,
    y_train
)

xgb_pred = xgb_model.predict(
    X_test
)

xgb_prob = xgb_model.predict_proba(
    X_test
)[:, 1]

xgb_result = advanced_evaluate_model(
    'XGBoost Baseline',
    y_test,
    xgb_pred,
    xgb_prob
)

pd.DataFrame([xgb_result])

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

In [ ]:
 rf_tuning_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),

        ('classifier',
         RandomForestClassifier(
             random_state=42,
             n_jobs=-1
         )
        )
    ]
)

In [ ]:
 rf_param_grid_fast = {

    'classifier__n_estimators': [
        150,
        250,
        350
    ],

    'classifier__max_depth': [
        8,
        12,
        16
    ],

    'classifier__min_samples_leaf': [
        5,
        10,
        20
    ],

    'classifier__max_features': [
        'sqrt'
    ]
}

In [ ]:
 from sklearn.model_selection import TimeSeriesSplit

time_cv_fast = TimeSeriesSplit(
    n_splits=3
)

In [ ]:
 rf_search = RandomizedSearchCV(

    estimator=rf_tuning_pipeline,

    param_distributions=rf_param_grid_fast,

    n_iter=8,

    scoring='roc_auc',

    cv=time_cv_fast,

    random_state=42,

    verbose=1,

    n_jobs=-1
)

In [ ]:
 rf_search.fit(
    X_train,
    y_train
)

In [ ]:
print("Best Random Forest Parameters:")
print(rf_search.best_params_)

print("\nBest Cross-Validation ROC-AUC:")
print(rf_search.best_score_)

In [ ]:
best_rf_model = rf_search.best_estimator_

In [ ]:
best_rf_pred = best_rf_model.predict(
    X_test
)

best_rf_prob = best_rf_model.predict_proba(
    X_test
)[:, 1]

In [ ]:
rf_fast_result = advanced_evaluate_model(
    'Tuned Random Forest',
    y_test,
    best_rf_pred,
    best_rf_prob
)

pd.DataFrame([rf_fast_result])

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb_tuning_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),

        ('classifier',
         XGBClassifier(
             eval_metric='logloss',
             random_state=42,
             n_jobs=-1,
             tree_method='hist'
         )
        )
    ]
)

In [ ]:
tree_method='hist'

In [ ]:
xgb_param_grid_fast = {

    'classifier__n_estimators': [
        150,
        250,
        350
    ],

    'classifier__max_depth': [
        3,
        5,
        7
    ],

    'classifier__learning_rate': [
        0.03,
        0.05,
        0.1
    ],

    'classifier__subsample': [
        0.8,
        1.0
    ],

    'classifier__colsample_bytree': [
        0.8,
        1.0
    ]
}

In [ ]:
xgb_search = RandomizedSearchCV(

    estimator=xgb_tuning_pipeline,

    param_distributions=xgb_param_grid_fast,

    n_iter=8,

    scoring='roc_auc',

    cv=time_cv_fast,

    random_state=42,

    verbose=1,

    n_jobs=-1
)

In [ ]:
xgb_search.fit(
    X_train,
    y_train
)

In [ ]:
print("Best XGBoost Parameters:")
print(xgb_search.best_params_)

print("\nBest Cross-Validation ROC-AUC:")
print(xgb_search.best_score_)

In [ ]:
best_xgb_model = xgb_search.best_estimator_

In [ ]:
best_xgb_pred = best_xgb_model.predict(
    X_test
)

best_xgb_prob = best_xgb_model.predict_proba(
    X_test
)[:, 1]

In [ ]:
xgb_fast_result = advanced_evaluate_model(
    'Tuned XGBoost',
    y_test,
    best_xgb_pred,
    best_xgb_prob
)

pd.DataFrame([xgb_fast_result])

In [ ]:
advanced_results = [

    advanced_evaluate_model(
        'Tuned Random Forest',
        y_test,
        best_rf_pred,
        best_rf_prob
    ),

    advanced_evaluate_model(
        'Tuned XGBoost',
        y_test,
        best_xgb_pred,
        best_xgb_prob
    )
]

advanced_results_df = pd.DataFrame(
    advanced_results
)

advanced_results_df.sort_values(
    'ROC_AUC',
    ascending=False
)

In [ ]:
threshold_results = []

for threshold in np.arange(
    0.30,
    0.71,
    0.05
):

    predictions = (
        best_xgb_prob >= threshold
    ).astype(int)

    threshold_results.append({

        'Threshold':
            round(threshold, 2),

        'Accuracy':
            accuracy_score(
                y_test,
                predictions
            ),

        'Precision':
            precision_score(
                y_test,
                predictions
            ),

        'Recall':
            recall_score(
                y_test,
                predictions
            ),

        'F1':
            f1_score(
                y_test,
                predictions
            )
    })

In [ ]:
threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df

In [ ]:
threshold_df.sort_values(
    'F1',
    ascending=False
).head()

In [ ]:
precision, recall, thresholds = (
    precision_recall_curve(
        y_test,
        best_xgb_prob
    )
)

plt.figure(figsize=(8, 6))

plt.plot(
    recall,
    precision
)

plt.xlabel('Recall')
plt.ylabel('Precision')

plt.title(
    'Precision-Recall Curve - Tuned XGBoost'
)

plt.show()

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

In [ ]:
calibrated_xgb = CalibratedClassifierCV(
    estimator=best_xgb_model,
    method='sigmoid',
    cv=2
)
calibrated_xgb.fit(
    X_train,
    y_train
)

calibrated_prob = (
    calibrated_xgb
    .predict_proba(X_test)[:, 1]
)

calibrated_pred = (
    calibrated_prob >= 0.50
).astype(int)

calibrated_result = advanced_evaluate_model(
    'Calibrated XGBoost',
    y_test,
    calibrated_pred,
    calibrated_prob
)

pd.DataFrame(
    [calibrated_result]
)

In [ ]:
import shap

In [ ]:
final_preprocessor = (
    best_xgb_model
    .named_steps['preprocessor']
)

In [ ]:
X_test_transformed = (
    final_preprocessor
    .transform(X_test)
)

sample_size = min(
    500,
    X_test_transformed.shape[0]
)

X_shap = X_test_transformed[
    :sample_size
]

In [ ]:
if hasattr(X_shap, 'toarray'):
    X_shap = X_shap.toarray()

In [ ]:
encoded_feature_names = (
    final_preprocessor
    .named_transformers_['cat']
    .named_steps['encoder']
    .get_feature_names_out(
        categorical_features
    )
)

all_feature_names = (
    numerical_features +
    list(encoded_feature_names)
)

In [ ]:
xgb_classifier = (
    best_xgb_model
    .named_steps['classifier']
)

In [ ]:
explainer = shap.TreeExplainer(
    xgb_classifier
)

shap_values = explainer.shap_values(
    X_shap
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=all_feature_names,
    max_display=20
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=all_feature_names,
    plot_type='bar',
    max_display=20
)

In [ ]:
train_prob = (
    best_xgb_model
    .predict_proba(X_train)[:, 1]
)

test_prob = (
    best_xgb_model
    .predict_proba(X_test)[:, 1]
)

In [ ]:
train_auc = roc_auc_score(
    y_train,
    train_prob
)

test_auc = roc_auc_score(
    y_test,
    test_prob
)

print(
    "Training ROC-AUC:",
    round(train_auc, 4)
)

print(
    "Testing ROC-AUC:",
    round(test_auc, 4)
)

print(
    "AUC Difference:",
    round(train_auc - test_auc, 4)
)

In [ ]:
final_results = [

    advanced_evaluate_model(
        'Tuned Random Forest',
        y_test,
        best_rf_pred,
        best_rf_prob
    ),

    advanced_evaluate_model(
        'Tuned XGBoost',
        y_test,
        best_xgb_pred,
        best_xgb_prob
    ),

    advanced_evaluate_model(
        'Calibrated XGBoost',
        y_test,
        calibrated_pred,
        calibrated_prob
    )
]

final_comparison = pd.DataFrame(
    final_results
)

final_comparison.sort_values(
    'ROC_AUC',
    ascending=False
)

In [ ]:
final_results = [

    advanced_evaluate_model(
        'Tuned Random Forest',
        y_test,
        best_rf_pred,
        best_rf_prob
    ),

    advanced_evaluate_model(
        'Tuned XGBoost',
        y_test,
        best_xgb_pred,
        best_xgb_prob
    ),

    advanced_evaluate_model(
        'Calibrated XGBoost',
        y_test,
        calibrated_pred,
        calibrated_prob
    )
]

final_comparison = pd.DataFrame(
    final_results
)

final_comparison.sort_values(
    'ROC_AUC',
    ascending=False
)

In [ ]:
n_iter=5

In [ ]:
n_iter=5

In [ ]:
cv=time_cv_fast

In [ ]:
best_rf_model
best_xgb_model
calibrated_xgb

best_rf_pred
best_rf_prob

best_xgb_pred
best_xgb_prob

calibrated_pred
calibrated_prob

threshold_df
order_level
test_mask

In [ ]:
final_results = [

    advanced_evaluate_model(
        'Tuned Random Forest',
        y_test,
        best_rf_pred,
        best_rf_prob
    ),

    advanced_evaluate_model(
        'Tuned XGBoost',
        y_test,
        best_xgb_pred,
        best_xgb_prob
    ),

    advanced_evaluate_model(
        'Calibrated XGBoost',
        y_test,
        calibrated_pred,
        calibrated_prob
    )
]

final_comparison = pd.DataFrame(final_results)

final_comparison

In [ ]:
final_comparison.sort_values(
    by='ROC_AUC',
    ascending=False
)

In [ ]:
final_comparison.sort_values(
    by='F1',
    ascending=False
)

In [ ]:
final_comparison.sort_values(
    by='Recall',
    ascending=False
)

In [ ]:
metrics_to_plot = [
    'Accuracy',
    'Precision',
    'Recall',
    'F1',
    'ROC_AUC',
    'PR_AUC'
]

final_comparison.set_index(
    'Model'
)[metrics_to_plot].plot(
    kind='bar',
    figsize=(12, 6)
)

plt.title('Advanced Model Performance Comparison')
plt.ylabel('Score')
plt.xticks(rotation=25)
plt.legend(loc='lower right')
plt.show()

In [ ]:
fpr_rf, tpr_rf, _ = roc_curve(
    y_test,
    best_rf_prob
)

fpr_xgb, tpr_xgb, _ = roc_curve(
    y_test,
    best_xgb_prob
)

fpr_cal, tpr_cal, _ = roc_curve(
    y_test,
    calibrated_prob
)

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(
    fpr_rf,
    tpr_rf,
    label='Tuned Random Forest'
)

plt.plot(
    fpr_xgb,
    tpr_xgb,
    label='Tuned XGBoost'
)

plt.plot(
    fpr_cal,
    tpr_cal,
    label='Calibrated XGBoost'
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle='--'
)

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.show()

In [ ]:
precision_rf, recall_rf, _ = precision_recall_curve(
    y_test,
    best_rf_prob
)

precision_xgb, recall_xgb, _ = precision_recall_curve(
    y_test,
    best_xgb_prob
)

precision_cal, recall_cal, _ = precision_recall_curve(
    y_test,
    calibrated_prob
)

In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(
    recall_rf,
    precision_rf,
    label='Random Forest'
)

plt.plot(
    recall_xgb,
    precision_xgb,
    label='XGBoost'
)

plt.plot(
    recall_cal,
    precision_cal,
    label='Calibrated XGBoost'
)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve Comparison')
plt.legend()
plt.show()

In [ ]:
threshold_df.sort_values(
    by='F1',
    ascending=False
)

In [ ]:
best_threshold_row = (
    threshold_df
    .sort_values(
        by='F1',
        ascending=False
    )
    .iloc[0]
)

best_threshold = (
    best_threshold_row['Threshold']
)

print(
    "Best threshold based on F1:",
    best_threshold
)

print(best_threshold_row)

In [ ]:
high_recall_options = (
    threshold_df[
        threshold_df['Recall'] >= 0.75
    ]
    .sort_values(
        by='Precision',
        ascending=False
    )
)

high_recall_options

In [ ]:
high_recall_options = (
    threshold_df[
        threshold_df['Recall'] >= 0.70
    ]
    .sort_values(
        by='Precision',
        ascending=False
    )
)

high_recall_options

In [ ]:
print(
    "XGBoost Brier Score:",
    brier_score_loss(
        y_test,
        best_xgb_prob
    )
)

print(
    "Calibrated XGBoost Brier Score:",
    brier_score_loss(
        y_test,
        calibrated_prob
    )
)

In [ ]:
final_probabilities = calibrated_prob
final_model_name = 'Calibrated XGBoost'

In [ ]:
final_probabilities = best_xgb_prob
final_model_name = 'Tuned XGBoost'

In [ ]:
print(
    "Final probability model:",
    final_model_name
)

In [ ]:
final_predictions = (
    final_probabilities
    >= best_threshold
).astype(int)

In [ ]:
print(
    classification_report(
        y_test,
        final_predictions
    )
)

In [ ]:
final_cm = confusion_matrix(
    y_test,
    final_predictions
)

print(final_cm)

In [ ]:
tn, fp, fn, tp = final_cm.ravel()

print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay(
    confusion_matrix=final_cm,
    display_labels=[
        'Not Late',
        'Late'
    ]
).plot()

plt.title(
    'Final Model Confusion Matrix'
)

plt.show()

In [ ]:
def assign_risk_level(probability):

    if probability < 0.40:
        return 'Low Risk'

    elif probability < 0.70:
        return 'Medium Risk'

    else:
        return 'High Risk'

In [ ]:
risk_levels = [
    assign_risk_level(p)
    for p in final_probabilities
]

pd.Series(
    risk_levels
).value_counts()

In [ ]:
final_scored_data = (
    order_level.loc[
        test_mask,
        [
            'Order Id',
            'Order_Date',
            'Type',
            'Customer Segment',
            'Market',
            'Order Country',
            'Order Region',
            'Shipping Mode',
            'Sales',
            'Order Profit Per Order',
            'Number_of_Items',
            'Number_of_Products',
            'Number_of_Categories',
            'Late_delivery_risk'
        ]
    ]
    .copy()
)

In [ ]:
final_scored_data[
    'Predicted_Late_Delivery'
] = final_predictions

final_scored_data[
    'Late_Delivery_Probability'
] = final_probabilities

final_scored_data[
    'Risk_Level'
] = risk_levels

final_scored_data[
    'Prediction_Model'
] = final_model_name

final_scored_data[
    'Decision_Threshold'
] = best_threshold

final_scored_data.head()

In [ ]:
print(
    "Rows:",
    len(final_scored_data)
)

print(
    "\nActual target:"
)

print(
    final_scored_data[
        'Late_delivery_risk'
    ].value_counts()
)

print(
    "\nPredicted target:"
)

print(
    final_scored_data[
        'Predicted_Late_Delivery'
    ].value_counts()
)

print(
    "\nRisk levels:"
)

print(
    final_scored_data[
        'Risk_Level'
    ].value_counts()
)

In [ ]:
risk_validation = pd.crosstab(
    final_scored_data[
        'Risk_Level'
    ],
    final_scored_data[
        'Late_delivery_risk'
    ],
    normalize='index'
) * 100

risk_validation.round(2)

In [ ]:
risk_summary = (
    final_scored_data
    .groupby(
        'Risk_Level',
        observed=True
    )
    .agg(
        Orders=('Order Id', 'count'),
        Average_Predicted_Risk=(
            'Late_Delivery_Probability',
            'mean'
        ),
        Actual_Late_Rate=(
            'Late_delivery_risk',
            'mean'
        )
    )
    .reset_index()
)

In [ ]:
risk_summary[
    'Average_Predicted_Risk'
] *= 100

risk_summary[
    'Actual_Late_Rate'
] *= 100

risk_summary

In [ ]:
top_high_risk_orders = (
    final_scored_data
    .sort_values(
        by='Late_Delivery_Probability',
        ascending=False
    )
    .head(20)
)

top_high_risk_orders

In [ ]:
market_risk_summary = (
    final_scored_data
    .groupby('Market')
    .agg(
        Total_Orders=(
            'Order Id',
            'count'
        ),

        Actual_Late_Rate=(
            'Late_delivery_risk',
            'mean'
        ),

        Average_Predicted_Risk=(
            'Late_Delivery_Probability',
            'mean'
        )
    )
    .reset_index()
)

In [ ]:
market_risk_summary[
    'Actual_Late_Rate'
] *= 100

market_risk_summary[
    'Average_Predicted_Risk'
] *= 100

market_risk_summary

In [ ]:
region_risk_summary = (
    final_scored_data
    .groupby('Order Region')
    .agg(
        Total_Orders=(
            'Order Id',
            'count'
        ),

        Actual_Late_Rate=(
            'Late_delivery_risk',
            'mean'
        ),

        Average_Predicted_Risk=(
            'Late_Delivery_Probability',
            'mean'
        )
    )
    .reset_index()
)

In [ ]:
region_risk_summary[
    'Actual_Late_Rate'
] *= 100

region_risk_summary[
    'Average_Predicted_Risk'
] *= 100

region_risk_summary.sort_values(
    by='Average_Predicted_Risk',
    ascending=False
)

In [ ]:
shipping_risk_summary = (
    final_scored_data
    .groupby('Shipping Mode')
    .agg(
        Total_Orders=(
            'Order Id',
            'count'
        ),

        Actual_Late_Rate=(
            'Late_delivery_risk',
            'mean'
        ),

        Average_Predicted_Risk=(
            'Late_Delivery_Probability',
            'mean'
        )
    )
    .reset_index()
)

In [ ]:
shipping_risk_summary[
    'Actual_Late_Rate'
] *= 100

shipping_risk_summary[
    'Average_Predicted_Risk'
] *= 100

shipping_risk_summary

In [ ]:
final_comparison.to_csv(
    'DataCo_Final_Model_Comparison.csv',
    index=False
)

In [ ]:
threshold_df.to_csv(
    'DataCo_Threshold_Analysis.csv',
    index=False
)

In [ ]:
final_scored_data.to_csv(
    'DataCo_Final_Scored_Orders.csv',
    index=False
)

In [ ]:
risk_summary.to_csv(
    'DataCo_Risk_Level_Summary.csv',
    index=False
)

In [ ]:
market_risk_summary.to_csv(
    'DataCo_Market_Risk_Summary.csv',
    index=False
)

In [ ]:
shipping_risk_summary.to_csv(
    'DataCo_Shipping_Mode_Risk_Summary.csv',
    index=False
)

region_risk_summary.to_csv(
    'DataCo_Region_Risk_Summary.csv',
    index=False
)

top_high_risk_orders.to_csv(
    'DataCo_Top_High_Risk_Orders.csv',
    index=False
)



In [ ]:
import joblib

In [ ]:
joblib.dump(
    best_rf_model,
    'DataCo_Tuned_Random_Forest.pkl'
)

In [ ]:
 joblib.dump(
    best_xgb_model,
    'DataCo_Tuned_XGBoost.pkl'
)

In [ ]:
joblib.dump(
    calibrated_xgb,
    'DataCo_Calibrated_XGBoost.pkl'
)

In [ ]:
project_summary = pd.DataFrame({

    'Item': [
        'Problem Type',
        'Target',
        'Training Period',
        'Test Period',
        'Final Model',
        'Final Threshold',
        'Training Orders',
        'Testing Orders'
    ],

    'Value': [
        'Binary Classification',
        'Late_delivery_risk',
        '2015-2017',
        '2018',
        final_model_name,
        best_threshold,
        len(X_train),
        len(X_test)
    ]
})

In [ ]:
project_summary

In [ ]:
project_summary.to_csv(
    'DataCo_Project_Model_Summary.csv',
    index=False
)

In [ ]:
import os

generated_files = [
    'DataCo_Final_Model_Comparison.csv',
    'DataCo_Threshold_Analysis.csv',
    'DataCo_Final_Scored_Orders.csv',
    'DataCo_Risk_Level_Summary.csv',
    'DataCo_Market_Risk_Summary.csv',
    'DataCo_Region_Risk_Summary.csv',
    'DataCo_Shipping_Mode_Risk_Summary.csv',
    'DataCo_Top_High_Risk_Orders.csv',
    'DataCo_Project_Model_Summary.csv',
    'DataCo_Tuned_Random_Forest.pkl',
    'DataCo_Tuned_XGBoost.pkl',
    'DataCo_Calibrated_XGBoost.pkl'
]

for file in generated_files:
    print(
        file,
        "->",
        os.path.exists(file)
    )

In [ ]:
from google.colab import files

In [ ]:
files.download(
    'DataCo_Final_Scored_Orders.csv'
)

In [ ]:
files.download(
    'DataCo_Final_Model_Comparison.csv'
)

In [ ]:
files.download(
    'DataCo_Risk_Level_Summary.csv'
)

In [ ]:
files.download(
    'DataCo_Tuned_XGBoost.pkl'
)

In [ ]:
order_level.to_csv(
    'DataCo_Final_Order_Level_Dataset.csv',
    index=False
)

In [ ]:
files.download(
    'DataCo_Final_Order_Level_Dataset.csv'
)

# New Section